# tiny-log-parser — live demo

A Qwen3-4B fine-tune plus a deterministic epoch pre-pass, normalizing messy log
lines into a canonical 7-field JSON record. **100% vs 83.5%** exact match against
`gemini-3.1-pro-preview` on a 200-example held-out test set.

**Runtime → Change runtime type → T4 GPU**, then Runtime → Run all.

Repo: https://github.com/arshirazi97/tiny-log-parser

## 1. Setup (~2 min)

In [2]:
import os
os.chdir('/content')
!rm -rf tiny-log-parser
!git clone -q https://github.com/arshirazi97/tiny-log-parser.git
os.chdir('/content/tiny-log-parser')
!pip install -q "transformers==4.51.3" "peft==0.17.1" accelerate bitsandbytes openai

import transformers
if transformers.__version__ != "4.51.3":
    print("restarting to pick up new versions...")
    os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Load the model
Base weights (3.5 GB) plus the LoRA adapter (132 MB).

In [3]:
import torch, json, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from eval import build_prompt, parse, FIELDS
from score_hybrid import epoch_override

BASE = 'unsloth/qwen3-4b-unsloth-bnb-4bit'
tok = AutoTokenizer.from_pretrained(BASE, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE, device_map='auto'),
    'arshirazi/tiny-log-parser').eval()
print('ready')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'lora_ga_config', 'monteclora_config', 'peft_version', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/132M [00:00<?, ?B/s]

ready


## 3. The pipeline
The pre-pass fires only on bare epoch integers — the one subtask the model
reliably fails. Everything else goes to the model.

In [4]:
def normalize(lines, batch=8):
    out = []
    for i in range(0, len(lines), batch):
        chunk = lines[i:i+batch]
        enc = tok([build_prompt(l, []) for l in chunk],
                  return_tensors='pt', padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=120, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        for l, o in zip(chunk, gen):
            p = parse(tok.decode(o[enc.input_ids.shape[-1]:], skip_special_tokens=True))
            iso = epoch_override(l)
            if p and iso: p = {**p, 'timestamp': iso}
            out.append(p)
    return out

def show(lines):
    t0 = time.time(); preds = normalize(lines); dt = time.time()-t0
    for l, p in zip(lines, preds):
        print('\nIN  ', l[:96], '  [pre-pass]' if epoch_override(l) else '')
        print('OUT ', json.dumps(p) if p else '[unparseable]')
    print(f'\n{len(lines)} lines in {dt:.1f}s ({dt/len(lines)*1000:.0f} ms/line)')

## 4. Six formats, one schema

In [5]:
show([
  '<131>Mar  5 02:10:12 web-07 payments[4471]: TLS handshake aborted by peer [tid=8f2c91aa04bd77e35c1d6b0392ef4a18] took=4.775s',
  '10.14.2.9 - - [22/Jul/2026:09:15:44 +0500] "GET /api/v2/orders HTTP/1.1" 503 812 "-" "curl/8.4.0" rt=7.881',
  'ts=1780543196 level=warn service=inventory msg="stock below threshold" trace=7f3b1c2d4e5a6b8c9d0e1f2a3b4c5d6e latency=340ms',
  '2026-06-18 14:22:09,331 ERROR [payment-worker-3] c.a.p.RefundService - refund gateway timeout traceId=b2e4f6a8c0d2e4f6a8b0c2d4e6f8a0b2 elapsed=2140ms',
  '{"time":1783402811,"severity":"CRITICAL","container":"billing","message":"ledger write failed","duration_us":9120000}',
  '[2026-07-15T03:44:12+05:00] [FATAL] [search-indexer] shard replication halted (tid=e9d8c7b6a5f4e3d2c1b0a9f8e7d6c5b4) 15.2s',
])

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



IN   <131>Mar  5 02:10:12 web-07 payments[4471]: TLS handshake aborted by peer [tid=8f2c91aa04bd77e35 
OUT  {"timestamp": "2026-03-05T02:10:12Z", "level": "ERROR", "service": "payments", "trace_id": "8f2c91aa04bd77e35c1d6b0392ef4a18", "status_code": null, "latency_ms": 4775, "message": "TLS handshake aborted by peer"}

IN   10.14.2.9 - - [22/Jul/2026:09:15:44 +0500] "GET /api/v2/orders HTTP/1.1" 503 812 "-" "curl/8.4.0 
OUT  {"timestamp": "2026-07-22T04:15:44Z", "level": "ERROR", "service": "orders", "trace_id": null, "status_code": 503, "latency_ms": 7881, "message": "GET /api/v2/orders"}

IN   ts=1780543196 level=warn service=inventory msg="stock below threshold" trace=7f3b1c2d4e5a6b8c9d0   [pre-pass]
OUT  {"timestamp": "2026-06-04T03:19:56Z", "level": "WARNING", "service": "inventory", "trace_id": "7f3b1c2d4e5a6b8c9d0e1f2a3b4c5d6e", "status_code": null, "latency_ms": 340, "message": "stock below threshold"}

IN   2026-06-18 14:22:09,331 ERROR [payment-worker-3] c.a.p.RefundService 

## 5. Why the pre-pass exists
The model gets minutes and seconds right and the date wrong — epoch → calendar
arithmetic is integer division it can't do. Scaling training data 5k → 20k moved
this 0.5 points, so it was routed to `datetime.fromtimestamp()` instead.

In [6]:
line = 'ts=1780543196 level=warn service=inventory msg="stock below threshold" latency=340ms'
enc = tok(build_prompt(line, []), return_tensors='pt').to(model.device)
g = model.generate(**enc, max_new_tokens=120, do_sample=False, pad_token_id=tok.pad_token_id)
raw = parse(tok.decode(g[0][enc.input_ids.shape[-1]:], skip_special_tokens=True))
print('model alone :', raw.get('timestamp'))
print('pre-pass    :', epoch_override(line))

model alone : 2026-05-30T19:19:56Z
pre-pass    : 2026-06-04T03:19:56Z


## 6. Your own logs
Paste any lines below. Formats outside the six the model was trained on will
degrade — see Limitations in the README.

In [7]:
show([
  '<134>Jul 12 18:44:03 api-02 checkout[992]: order placed successfully [tid=c41d9a7e6b28f05134ae8d90bb17e2c6] took=0.234s',
])


IN   <134>Jul 12 18:44:03 api-02 checkout[992]: order placed successfully [tid=c41d9a7e6b28f05134ae8d 
OUT  {"timestamp": "2026-07-12T18:44:03Z", "level": "WARNING", "service": "checkout", "trace_id": "c41d9a7e6b28f05134ae8d90bb17e2c6", "status_code": null, "latency_ms": 234, "message": "order placed successfully"}

1 lines in 17.8s (17774 ms/line)


In [9]:
# ---- Optional: compare against the frontier baseline ----
# Needs an OpenRouter key (openrouter.ai/keys). ~$0.04 for these 3 lines.
# Leave blank and press Enter to skip.

import os, json, getpass
import pandas as pd
from IPython.display import display, HTML
from openai import OpenAI

key = getpass.getpass('OpenRouter API key (blank to skip): ').strip()

if not key:
    print('Skipped. Full 200-example comparison is in the README and baseline.json.')
else:
    client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=key)

    # The README baseline gets 3 few-shot examples. train.jsonl isn't in the repo,
    # so this runs Gemini zero-shot -- slightly harsher than the reported 83.5%.
    shots = []
    if os.path.exists('train.jsonl'):
        shots = [json.loads(l) for l in open('train.jsonl')][:3]

    lines = [
      '<131>Mar  5 02:10:12 web-07 payments[4471]: TLS handshake aborted by peer [tid=8f2c91aa04bd77e35c1d6b0392ef4a18] took=4.775s',
      'ts=1780543196 level=warn service=inventory msg="stock below threshold" trace=7f3b1c2d4e5a6b8c9d0e1f2a3b4c5d6e latency=340ms',
      '10.14.2.9 - - [22/Jul/2026:09:15:44 +0500] "GET /api/v2/orders HTTP/1.1" 503 812 "-" "curl/8.4.0" rt=7.881',
    ]

    ours = normalize(lines)   # defined in cell 3, includes the epoch pre-pass

    for line, a in zip(lines, ours):
        r = client.chat.completions.create(
            model='google/gemini-3.1-pro-preview', temperature=0, max_tokens=8000,
            messages=[{'role': 'user', 'content': build_prompt(line, shots)}])
        b = parse(r.choices[0].message.content)

        df = pd.DataFrame({
            'ours':   [(a or {}).get(f) for f in FIELDS],
            'gemini': [(b or {}).get(f) for f in FIELDS],
        }, index=FIELDS)
        diff = [x != y for x, y in zip(df['ours'], df['gemini'])]

        display(HTML(f"<pre style='margin:16px 0 4px;white-space:pre-wrap'>"
                     f"<b>IN</b>  {line[:120]}</pre>"))
        display(df.style
                  .set_properties(**{'text-align': 'left'})
                  .apply(lambda s: ['background-color:#fff3cd' if d else '' for d in diff],
                         axis=0))

    print('\nFull 200-example scoring: see the Results table in the README.')

OpenRouter API key (blank to skip): ··········


,ours,gemini
timestamp,2026-03-05T02:10:12Z,2026-03-05T02:10:12Z
level,ERROR,ERROR
service,payments,payments
trace_id,8f2c91aa04bd77e35c1d6b0392ef4a18,8f2c91aa04bd77e35c1d6b0392ef4a18
status_code,None,None
latency_ms,4775,4775
message,TLS handshake aborted by peer,TLS handshake aborted by peer


,ours,gemini
timestamp,2026-06-04T03:19:56Z,2026-06-04T03:19:56Z
level,WARNING,WARNING
service,inventory,inventory
trace_id,7f3b1c2d4e5a6b8c9d0e1f2a3b4c5d6e,7f3b1c2d4e5a6b8c9d0e1f2a3b4c5d6e
status_code,None,None
latency_ms,340,340
message,stock below threshold,stock below threshold


,ours,gemini
timestamp,2026-07-22T04:15:44Z,2026-07-22T04:15:44Z
level,ERROR,ERROR
service,orders,unknown
trace_id,None,None
status_code,503,503
latency_ms,7881,7881
message,GET /api/v2/orders,GET /api/v2/orders



Full 200-example scoring: see the Results table in the README.
